# C++ and CUDA: actual measurements

The AI is trained in Python. Inference runs in `native/cpu.cpp` and `native/gpu.cu`. Use CPU-only mode on machines without a compatible NVIDIA GPU. Use `CUDA_Colab.ipynb` to compare CPU and CUDA on the same remote machine. Timings come from hardware execution.

In [ ]:
from pathlib import Path
import sys

# Locate the project from its root or notebooks directory.
ROOT = Path.cwd()
if not (ROOT / "python" / "ai.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "python"))

from IPython.display import Image, display, Markdown
from ai import MLP
from analysis import run_benchmark, analyze
%matplotlib inline


## 1. Compile CPU code
The helper uses an installed C++ compiler. Zig is also supported for CPU-only builds. Native inference is separate from training and analysis.

In [ ]:
from build_native import build

# Compilation is excluded from the measured timings.
binary = build(cpu_only=True)


## 2. Full inference and numerical parity
Use identical FP32 weights and inputs exported from Python. The executable tests rectangular matrix products and checks final probabilities against the reference. It then performs five warmups and 21 repetitions per batch.

In [ ]:
path = run_benchmark(binary, ROOT / "results" / "native", batches=[1, 8, 64, 128, 256, 1024], cpu_only=True)
rows = analyze(path)
display(Image(filename=str(path.parent / "comparison_real.png")))
display(Markdown((path.parent / "analysis.md").read_text(encoding="utf-8")))


## 3. What to compare when a GPU is available

- `device_events`: interval between CUDA stream events for the whole network.
- `resident_wall`: wall time for resident-data inference and synchronization.
- `transfer_wall`: H2D + network + D2H + synchronization, with resident weights.

Compare CPU `wall` with GPU `transfer_wall` for input/output-inclusive cost. Use the same machine. Speedup above 1 means the GPU was faster in this measurement; no winner is predetermined.